In [1]:
import pandas as pd
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import torch.nn.functional as F
from pathlib import Path
import torch.nn as nn

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 200)

# ============================================================
# LOAD PRETRAINED RESNET18
# ============================================================
FINETUNED_MODEL_PATH = "resnet18_tinyimagenet_fc.pth" # <-- UPDATE THIS PATH
resnet.load_state_dict(torch.load(FINETUNED_MODEL_PATH, map_location=device))
resnet.to(device)

resnet.eval()

# ============================================================
# IMAGE TRANSFORM
# ============================================================
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# ============================================================
# LOAD IMAGENET CLASS MAPPING
# ============================================================
# import json
# import urllib.request

import os

TRAIN_DIR = "ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-train/tin_no_resize_sample_removed"

train_classes = sorted(os.listdir(TRAIN_DIR))
wnid_to_idx = {cls_name: i for i, cls_name in enumerate(train_classes)}
print(f"Locked mapping dictionary with {len(wnid_to_idx)} target classes.")

# def load_imagenet_mapping():
#     url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"

#     req = urllib.request.Request(
#         url,
#         headers={'User-Agent': 'Mozilla/5.0'}
#     )

#     with urllib.request.urlopen(req) as response:
#         class_idx = json.loads(response.read().decode("utf-8"))

#     return {v[0]: int(k) for k, v in class_idx.items()}

# wnid_to_idx = load_imagenet_mapping()

print(f"Loaded {len(wnid_to_idx)} ImageNet classes.")


# ============================================================
# LOAD TEST SET
# ============================================================
df = pd.read_csv("oracle_test_200 .csv")
# ============================================================
# BUILD THE EXACT 200-CLASS MASK (Using Training Data)
# ============================================================
# 1. Read your TRAINING dataset (which has all 200 classes)
# train_df = pd.read_csv("lens_train.csv") # Change to your actual train CSV name

# # 2. Extract the absolute list of 200 unique folder names
# target_200_classes = train_df['ae_image_path'].apply(lambda x: Path(x).parent.name).unique()

# 3. Convert those folder names into ImageNet integer indices
# valid_indices = [wnid_to_idx[c] for c in target_200_classes if c in wnid_to_idx]

# print(f"Masking enabled: Restricting predictions to exactly {len(valid_indices)} classes.")

# 4. Create a mask of negative infinity for all 1000 classes
# mask = torch.full((1000,), float('-inf')).to(device)

# 5. Set the valid 200 classes to 0.0
# mask[valid_indices] = 0.0

# ============================================================
# NOW LOAD AND EVALUATE THE TEST DATA
# ============================================================
# df = pd.read_csv("lens_test.csv")
df = df.iloc[:600].reset_index(drop=True)
print(f"Evaluating on {len(df)} test samples.")

# ============================================================
# EVALUATION
# ============================================================
top1_correct = 0
top5_correct = 0
total = len(df)

for i, row in df.iterrows():

    csv_path = Path(row["image_path"])
    
    # 2. Extract the required metadata
    class_name = csv_path.parent.name
    filename = csv_path.name
    lighting = row["lighting"] # Make sure this column exists in your new CSV!
    
    # 3. Navigate backwards out of param_control/lX/param_X/class/file
    base_dir = csv_path.parents[4]
    
    # 4. REBUILD THE TRUE AUTO EXPOSURE PATH
    # (Adjust "auto_exposure" or "param_1" based on your exact folder structure)
    image_path = os.path.join('ImageNet-ES-Diverse/', base_dir, "auto_exposure", lighting, "param_1", class_name, filename)
    
    # 5. Load the true degraded image
    if not os.path.exists(image_path):
        continue


    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    # Folder name is the ImageNet class
    class_name = Path(image_path).parent.name

    if class_name not in wnid_to_idx:
        print("not")
        continue

    target = row["label"]
    # print(target)

    with torch.no_grad():

        output = resnet(image)

        # ---------------- Top-1 ----------------
        prediction = output.argmax(dim=1).item()

        if prediction == target:
            top1_correct += 1

        # ---------------- Top-5 ----------------
        top5 = torch.topk(output, k=5, dim=1).indices.squeeze(0)

        if target in top5:
            top5_correct += 1

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{total} images")

# ============================================================
# RESULTS
# ============================================================
top1_acc = 100 * top1_correct / total
top5_acc = 100 * top5_correct / total

print("\n===============================")
print("Baseline 1 : Auto Exposure")
print("===============================")
print(f"Images Evaluated : {total}")
print(f"Top-1 Accuracy   : {top1_acc:.2f}%")
print(f"Top-5 Accuracy   : {top5_acc:.2f}%")

Using device: cuda
Locked mapping dictionary with 200 target classes.
Loaded 200 ImageNet classes.
Evaluating on 600 test samples.
Processed 50/600 images
Processed 100/600 images
Processed 150/600 images
Processed 200/600 images
Processed 250/600 images
Processed 300/600 images
Processed 350/600 images
Processed 400/600 images
Processed 450/600 images
Processed 500/600 images
Processed 550/600 images
Processed 600/600 images

Baseline 1 : Auto Exposure
Images Evaluated : 600
Top-1 Accuracy   : 13.00%
Top-5 Accuracy   : 30.33%


In [4]:
import random

sample = df.sample(5, random_state=42)
for _, row in sample.iterrows():
    from pathlib import Path
    p = Path(row["ae_image_path"])
    class_name = p.parent.name
    print(class_name, class_name in wnid_to_idx, wnid_to_idx.get(class_name))

KeyError: 'ae_image_path'

In [2]:
import json
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import torch.nn as nn

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# LOAD PRETRAINED RESNET18
# ============================================================

resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 200)

# ============================================================
# LOAD PRETRAINED RESNET18
# ============================================================
FINETUNED_MODEL_PATH = "resnet18_tinyimagenet_fc.pth" # <-- UPDATE THIS PATH
resnet.load_state_dict(torch.load(FINETUNED_MODEL_PATH, map_location=device))
resnet.to(device)

resnet.eval()

# ============================================================
# IMAGE TRANSFORM
# ============================================================
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# ============================================================
# LOAD IMAGENET CLASS MAP
# ============================================================
import os

TRAIN_DIR = "ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-train/tin_no_resize_sample_removed"

train_classes = sorted(os.listdir(TRAIN_DIR))
wnid_to_idx = {cls_name: i for i, cls_name in enumerate(train_classes)}
print(f"Locked mapping dictionary with {len(wnid_to_idx)} target classes.")

print(f"Loaded {len(wnid_to_idx)} ImageNet classes.")

# ============================================================
# LOAD TEST CSV
# ============================================================
df = pd.read_csv("oracle_test_200 .csv")

# ============================================================
# BUILD THE EXACT 200-CLASS MASK (Using Training Data)
# ============================================================
# 1. Read your TRAINING dataset (which has all 200 classes)
# train_df = pd.read_csv("lens_train.csv") # Change to your actual train CSV name

# # 2. Extract the absolute list of 200 unique folder names
# target_200_classes = train_df['ae_image_path'].apply(lambda x: Path(x).parent.name).unique()

# # 3. Convert those folder names into ImageNet integer indices
# valid_indices = [wnid_to_idx[c] for c in target_200_classes if c in wnid_to_idx]

# print(f"Masking enabled: Restricting predictions to exactly {len(valid_indices)} classes.")

# # 4. Create a mask of negative infinity for all 1000 classes
# mask = torch.full((1000,), float('-inf')).to(device)

# # 5. Set the valid 200 classes to 0.0
# mask[valid_indices] = 0.0

# ============================================================
# NOW LOAD AND EVALUATE THE TEST DATA
# ============================================================
# df = pd.read_csv("lens_test.csv")
df = df.iloc[:600].reset_index(drop=True)
print(f"Evaluating on {len(df)} test samples.")

# ============================================================
# EVALUATION
# ============================================================
top1_correct = 0
top5_correct = 0

total = len(df)

for i, row in df.iterrows():

    csv_path = Path(row["image_path"])
    
    # 2. Extract the required metadata
    class_name = csv_path.parent.name
    filename = csv_path.name
    lighting = row["lighting"] # Make sure this column exists in your new CSV!
    
    # 3. Navigate backwards out of param_control/lX/param_X/class/file
    base_dir = csv_path.parents[4]
    
    # 4. REBUILD THE TRUE AUTO EXPOSURE PATH
    # (Adjust "auto_exposure" or "param_1" based on your exact folder structure)
    image_path = os.path.join('ImageNet-ES-Diverse/', base_dir, "auto_exposure", lighting, "param_1", class_name, filename)

    # Oracle parameter (convert back to 1-27)
    confidence_vector = np.fromstring(row['all_27_confidences'].strip("[]"), sep=",")
    oracle_param = int(torch.argmax(torch.tensor(confidence_vector))) + 1
    # print(oracle_param)

    # oracle_param = int(row["best_param_index"]) + 1

    light_env = row["lighting"]
    ae_path = Path(image_path)

    class_name = ae_path.parent.name
    filename = ae_path.name

    # Build oracle image path
    oracle_path = (
      ae_path.parents[4]
      / "param_control"
      / light_env
      / f"param_{oracle_param}"
      / class_name
      / filename
  )


    if not oracle_path.exists():
        continue

    image = Image.open(oracle_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    target = wnid_to_idx[class_name]

    with torch.no_grad():

        output = resnet(image)
        output = output 

        # Top-1
        prediction = output.argmax(dim=1).item()

        if prediction == target:
            top1_correct += 1

        # Top-5
        top5 = torch.topk(output, 5, dim=1).indices.squeeze(0)

        if target in top5:
            top5_correct += 1

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{total}")

# ============================================================
# RESULTS
# ============================================================
top1_acc = 100 * top1_correct / total
top5_acc = 100 * top5_correct / total

print("\n===============================")
print("Baseline 2 : Oracle Parameters")
print("===============================")
print(f"Images Evaluated : {total}")
print(f"Top-1 Accuracy   : {top1_acc:.2f}%")
print(f"Top-5 Accuracy   : {top5_acc:.2f}%")

Using device: cuda
Locked mapping dictionary with 200 target classes.
Loaded 200 ImageNet classes.
Evaluating on 600 test samples.
Processed 50/600
Processed 100/600
Processed 150/600
Processed 200/600
Processed 250/600
Processed 300/600
Processed 350/600
Processed 400/600
Processed 450/600
Processed 500/600
Processed 550/600
Processed 600/600

Baseline 2 : Oracle Parameters
Images Evaluated : 600
Top-1 Accuracy   : 46.83%
Top-5 Accuracy   : 76.83%


In [11]:
import json
import urllib.request
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ============================================================
# FEATURE EXTRACTOR (from utils.py) - frozen resnet18, fc removed
# ============================================================
_resnet_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
feature_extractor = nn.Sequential(*list(_resnet_backbone.children())[:-1]).to(device)
feature_extractor.eval()

for param in feature_extractor.parameters():
    param.requires_grad = False


def extract_features(image_tensor):
    """
    Extracts lighting-aware features from an AE image tensor (3, H, W).
    Returns a 1D tensor of shape (586,): 74 handcrafted + 512 deep features.
    """
    r, g, b = image_tensor[0], image_tensor[1], image_tensor[2]
    _, H, W = image_tensor.shape
    features = []

    # ── 1. RGB spatial histogram (60 dims) ─────────────────────────────
    grid = 2
    h_step, w_step = H // grid, W // grid
    for row in range(grid):
        for col in range(grid):
            ys, ye = row * h_step, (row + 1) * h_step
            xs, xe = col * w_step, (col + 1) * w_step
            n_pixels = h_step * w_step
            for channel in [r, g, b]:
                block = channel[ys:ye, xs:xe]
                hist = torch.histc(block, bins=5, min=0.0, max=1.0)
                features.append(hist / n_pixels)

    # ── 2. Per-channel global stats (9 dims) ────────────────────────────
    for channel in [r, g, b]:
        ch_mean = channel.mean()
        ch_std = channel.std()
        clipping = ((channel > 0.95) | (channel < 0.05)).float().mean()
        features.append(torch.stack([ch_mean, ch_std, clipping]))

    # ── 3. Local brightness contrast (5 dims) ────────────────────────────
    gray = 0.2989 * r + 0.5870 * g + 0.1140 * b
    block_means = []
    for row in range(grid):
        for col in range(grid):
            ys, ye = row * h_step, (row + 1) * h_step
            xs, xe = col * w_step, (col + 1) * w_step
            block_means.append(gray[ys:ye, xs:xe].mean())
    block_means_t = torch.stack(block_means)
    global_contrast = block_means_t.std()
    features.append(block_means_t)
    features.append(global_contrast.unsqueeze(0))

    # ── 4. Deep resnet18 features (512 dims) ─────────────────────────────
    with torch.no_grad():
        image_batch = image_tensor.unsqueeze(0).to(device)
        deep_features = feature_extractor(image_batch)
        deep_features = deep_features.flatten()

    handcrafted = torch.cat([f.flatten() for f in features])
    final = torch.cat([handcrafted, deep_features.cpu()])

    return final


# ============================================================
# LENS PREDICTOR MODEL (from models.py)
# ============================================================
class LensPredictorNetwork(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 27)
        )

    def forward(self, x):
        return self.network(x)


# ============================================================
# LOAD TRAINED LENS PREDICTOR
# ============================================================
INPUT_DIM = 586
MODEL_PATH = "resnet_histogram_3layer_v3.pth"   # change to your actual saved checkpoint filename

lens_model = LensPredictorNetwork(INPUT_DIM).to(device)

lens_model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device)
)

lens_model.eval()

print("Lens predictor loaded.")


# ============================================================
# LOAD IMAGENET RESNET18 (final evaluation classifier)
# ============================================================
resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 200)

# ============================================================
# LOAD PRETRAINED RESNET18
# ============================================================
FINETUNED_MODEL_PATH = "resnet18_tinyimagenet_fc.pth" # <-- UPDATE THIS PATH
resnet.load_state_dict(torch.load(FINETUNED_MODEL_PATH, map_location=device))
resnet.to(device)

resnet.eval()

print("ImageNet ResNet loaded.")


# ============================================================
# IMAGE TRANSFORM
# ============================================================
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# IMAGENET CLASS MAP
# ============================================================
import os

TRAIN_DIR = "ImageNet-ES-Diverse/ImageNet-ES-Diverse/es-train/tin_no_resize_sample_removed"

train_classes = sorted(os.listdir(TRAIN_DIR))
wnid_to_idx = {cls_name: i for i, cls_name in enumerate(train_classes)}
print(f"Locked mapping dictionary with {len(wnid_to_idx)} target classes.")

print(f"Loaded {len(wnid_to_idx)} ImageNet classes.")


# ============================================================
# LOAD TEST CSV - TAKE FIRST 600 IMAGES
# ============================================================
df = pd.read_csv("oracle_test_200 .csv")
df = df.iloc[:600].reset_index(drop=True)
# ============================================================
# BUILD THE EXACT 200-CLASS MASK (Using Training Data)
# ============================================================
# 1. Read your TRAINING dataset (which has all 200 classes)
# train_df = pd.read_csv("lens_train.csv") # Change to your actual train CSV name

# # 2. Extract the absolute list of 200 unique folder names
# target_200_classes = train_df['ae_image_path'].apply(lambda x: Path(x).parent.name).unique()

# # 3. Convert those folder names into ImageNet integer indices
# valid_indices = [wnid_to_idx[c] for c in target_200_classes if c in wnid_to_idx]

# print(f"Masking enabled: Restricting predictions to exactly {len(valid_indices)} classes.")

# # 4. Create a mask of negative infinity for all 1000 classes
# mask = torch.full((1000,), float('-inf')).to(device)

# # 5. Set the valid 200 classes to 0.0
# mask[valid_indices] = 0.0
# print(mask)

# ============================================================
# NOW LOAD AND EVALUATE THE TEST DATA
# ============================================================
# df = pd.read_csv("lens_test.csv")
# df = df.iloc[:600].reset_index(drop=True)
print(f"Evaluating on {len(df)} test samples.")

# ============================================================
# EVALUATION
# ============================================================
top1_correct = 0
top5_correct = 0
total_valid = 0

print("Evaluating Your Trained LensPredictor Pipeline...")

for i, row in df.iterrows():
    csv_path = Path(row["image_path"])
    
    # Extract metadata
    class_name = csv_path.parent.name
    filename = csv_path.name
    lighting = row["lighting"] 
    target = int(row["label"])
    
    # Get the inner base directory (e.g., 'ImageNet-ES-Diverse/es-diverse-test')
    base_dir = csv_path.parents[4]
    
    # 1. Build the TRUE AE Path (Prepending the outer ImageNet folder)
    ae_path = Path(os.path.join('ImageNet-ES-Diverse/', base_dir, "auto_exposure", lighting, "param_1", class_name, filename))
    
    if not ae_path.exists():
        continue

    # Load AE Image and Predict
    ae_image = Image.open(ae_path).convert("RGB")
    ae_tensor = transform(ae_image)

    feat = extract_features(ae_tensor).unsqueeze(0).to(device)
    with torch.no_grad():
        output = lens_model(feat)
        predicted_param = output.argmax(dim=1).item() +1

    # 2. Build the PREDICTED PARAM Path (CRITICAL FIX: Prepending the outer folder here too!)
    param_path = Path(os.path.join('ImageNet-ES-Diverse/', base_dir, "param_control", lighting, f"param_{predicted_param}", class_name, filename))

    if not param_path.exists():
        continue # If this triggers, the image file is actually missing from your hard drive

    # Load chosen image and classify
    img = Image.open(param_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = resnet(img_tensor)
        
        pred = logits.argmax(dim=1).item()
        # print(f"predicted {pred} ==> target {target}")
        if pred == target:
            top1_correct += 1

        top5 = torch.topk(logits, k=5, dim=1).indices.squeeze(0)
        if target in top5:
            top5_correct += 1

    # Increment counter and print progress
    total_valid += 1
    if total_valid % 50 == 0:
        print(f"Processed {total_valid} valid images...")

# ============================================================
# RESULTS
# ============================================================
print("\n===============================")
print("Your LensPredictor Pipeline")
print("===============================")
if total_valid == 0:
    print("ERROR: 0 images evaluated. Check if your param_path exists on your hard drive!")
else:
    print(f"Images Evaluated : {total_valid}")
    print(f"Top-1 Accuracy   : {100 * top1_correct / total_valid:.2f}%")
    print(f"Top-5 Accuracy   : {100 * top5_correct / total_valid:.2f}%")

Using device: cuda
Lens predictor loaded.
ImageNet ResNet loaded.
Locked mapping dictionary with 200 target classes.
Loaded 200 ImageNet classes.
Evaluating on 600 test samples.
Evaluating Your Trained LensPredictor Pipeline...
Processed 50 valid images...
Processed 100 valid images...
Processed 150 valid images...
Processed 200 valid images...
Processed 250 valid images...
Processed 300 valid images...
Processed 350 valid images...
Processed 400 valid images...
Processed 450 valid images...
Processed 500 valid images...
Processed 550 valid images...
Processed 600 valid images...

Your LensPredictor Pipeline
Images Evaluated : 600
Top-1 Accuracy   : 25.67%
Top-5 Accuracy   : 49.83%


lens approach vs our approach side by side - remove the candidate selection and make it - 27 x 200 just like oracle
lens 200 classes approach
masking approach - try out and find

In [ ]:
import pandas as pd
import torch
from PIL import Image
from pathlib import Path

# ... [Assume device, transforms, wnid_to_idx, and resnet are loaded] ...
# ... [Assume the 200-class 'mask' from the previous steps is already built] ...

print(f"Starting 27-Parameter Max-Confidence Evaluation on {len(df)} scenes...")

correct_predictions = 0
total_valid_scenes = 0
results_log = []

for i, row in df.iterrows():
    ae_path = Path(row["ae_image_path"])
    class_name = ae_path.parent.name
    filename = ae_path.name
    light_env = row["light_environment"]
    
    # Step 4: Find the original target class
    target_idx = wnid_to_idx.get(class_name)
    if target_idx is None:
        continue

    # Step 1: Load all 27 parameter images into a single batch
    batch_images = []
    loaded_params = [] # Keep track of which parameters successfully loaded
    
    for p in range(1, 28):
        param_path = (ae_path.parents[4] / "param_control" / light_env / 
                      f"param_{p}" / class_name / filename)
        
        if param_path.exists():
            img = Image.open(param_path).convert("RGB")
            img_tensor = transform(img)
            batch_images.append(img_tensor)
            loaded_params.append(p)
            
    # If no images loaded for this scene, skip it
    if len(batch_images) == 0:
        continue

    # Stack the list of 3D tensors into a single 4D batch tensor: [V, 3, 224, 224]
    # (Where V is the number of successfully loaded parameter images, ideally 27)
    batch_tensor = torch.stack(batch_images).to(device)

    with torch.no_grad():
        # Step 2: Pass all 27 images through the ResNet simultaneously
        # Output shape: [V, 1000]
        logits = resnet(batch_tensor)
        
        # Apply the 200-class mask to all 27 images at once
        # The 800 invalid classes drop to -infinity across the entire batch
        logits = logits + mask

        # Step 3: Find the global max value across the ENTIRE [V, 1000] tensor
        # We flatten the tensor to 1D to find the absolute highest single score
        flat_logits = logits.flatten()
        max_idx_1d = flat_logits.argmax().item()
        
        # Convert the 1D index back into 2D coordinates (Which image? Which class?)
        best_img_index = max_idx_1d // 1000
        predicted_class_idx = max_idx_1d % 1000
        
        # Map the image index back to the actual camera parameter (1-27)
        winning_param = loaded_params[best_img_index]

    # Check if the highest confidence class matches the ground truth
    is_correct = (predicted_class_idx == target_idx)
    
    if is_correct:
        correct_predictions += 1
        
    total_valid_scenes += 1

    # Log the data for deep analysis
    results_log.append({
        "image_name": filename,
        "target_class": class_name,
        "winning_param": winning_param,
        "predicted_class_idx": predicted_class_idx,
        "is_correct": is_correct
    })

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{len(df)} scenes...")

# Step 5: Find out how much percentage of the images have been correctly identified
final_accuracy = (correct_predictions / total_valid_scenes) * 100

print("\n=======================================")
print("MAX-CONFIDENCE (27-IMAGE) EVALUATION")
print("=======================================")
print(f"Total Scenes Evaluated : {total_valid_scenes}")
print(f"Total Correct          : {correct_predictions}")
print(f"Global Accuracy        : {final_accuracy:.2f}%")
print("=======================================")

Starting 27-Parameter Max-Confidence Evaluation on 600 scenes...
Processed 50/600 scenes...
Processed 100/600 scenes...
Processed 150/600 scenes...
Processed 200/600 scenes...
Processed 250/600 scenes...
Processed 300/600 scenes...
Processed 350/600 scenes...
Processed 400/600 scenes...
Processed 450/600 scenes...
Processed 500/600 scenes...
Processed 550/600 scenes...
Processed 600/600 scenes...

MAX-CONFIDENCE (27-IMAGE) EVALUATION
Total Scenes Evaluated : 600
Total Correct          : 216
Global Accuracy        : 36.00%


: 

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# =====================================================================
# 1. SYSTEM & DEVICE SETUP
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training executing on device: {device}")

# Reproducibility: same seed -> same shuffling, same crop augmentations,
# same weight init for the classifier head, run after run.
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def seed_worker(worker_id):
    worker_seed = SEED
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

# =====================================================================
# 2. DATASET LOADER (With CPU Optimization & Stratified Subsampling)
# =====================================================================
class LensCameraDataset(Dataset):
    def __init__(self, csv_file, transform=None, dataset_root='ImageNet-ES-Diverse', samples_per_class=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform
        self.dataset_root = dataset_root

        # --- STRATIFIED SUBSAMPLING TO REDUCE CPU LOAD ---
        # Extract the class folder name (e.g., 'n01440764') from ae_image_path
        self.df['derived_class'] = self.df['image_path'].apply(
            lambda x: x.replace('\\', '/').split('/')[-2]
        )
        
       

        num_classes_detected = self.df['derived_class'].nunique()
        print(f"Loaded {len(self.df):,} total images across {num_classes_detected} classes.")

        # --- CPU BOTTLENECK FIX: PRE-PARSE TARGETS IN MEMORY ---
        # Parses confidence strings once at startup instead of every epoch during training
        print("Pre-parsing target confidence vectors into memory...")
        self.targets = []
        for vec_str in self.df['all_27_confidences']:  # <--- Updated to match your new CSV
            cleaned = vec_str.replace('\n', '').strip('[]')
            confidences = np.fromstring(cleaned, sep=',')  # <--- Change sep=' ' to sep=','
            self.targets.append(torch.tensor(confidences, dtype=torch.float32))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Collapse duplicated directory path artifacts
        raw_path = row['image_path'].replace('\\', '/')
        raw_path = raw_path.replace(
            'ImageNet-ES-Diverse/ImageNet-ES-Diverse/',
            'ImageNet-ES-Diverse/'
        )
        image_path = os.path.join(self.dataset_root, raw_path)

        image = Image.open(image_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        # Return pre-parsed target tensor directly
        return image, self.targets[idx]

# =====================================================================
# 3. DATA AUGMENTATION & TRANSFORMS
# =====================================================================
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# =====================================================================
# 4. DATA LOADERS (Configured for CPU: 5 Train/Class, 1 Val/Class)
# =====================================================================
TRAIN_CSV_PATH = 'oracle_train_200.csv'
VAL_CSV_PATH = 'oracle_val_200.csv'

print("\n--- Initializing Training Dataset ---")
# 5 images per class * 200 classes = 1,000 images total (was 4,800)
train_dataset = LensCameraDataset(
    TRAIN_CSV_PATH, 
    transform=train_transform
)

print("\n--- Initializing Validation Dataset ---")
# 1 image per class * 200 classes = 200 images total (was 600)
val_dataset = LensCameraDataset(
    VAL_CSV_PATH, 
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =====================================================================
# 5. POLICY NETWORK MODEL ARCHITECTURE (image-only, no lighting branch)
# =====================================================================
class ImprovedLensPredictor(nn.Module):
    def __init__(self, num_classes=27):
        super().__init__()

        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        for param in self.backbone.parameters():
            param.requires_grad = False

        self.backbone.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256,128),

            nn.BatchNorm1d(128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128,num_classes)
              # Always outputs 27 parameters
        )

    def forward(self, img):
        image_features = self.backbone(img)
        return self.classifier(image_features)

# =====================================================================
# 6. MODEL INITIALIZATION
# =====================================================================
model = ImprovedLensPredictor(num_classes=27).to(device)

# =====================================================================
# 7. RANKNET LOSS, OPTIMIZER, SCHEDULER
# =====================================================================
def ranknet_loss(y_pred, y_true):
    pred_diffs = y_pred.unsqueeze(2) - y_pred.unsqueeze(1)
    pred_probs = torch.sigmoid(pred_diffs)

    true_diffs = y_true.unsqueeze(2) - y_true.unsqueeze(1)

    true_probs = torch.zeros_like(true_diffs)
    true_probs[true_diffs > 0] = 1.0   # i > j
    true_probs[true_diffs < 0] = 0.0   # i < j
    true_probs[true_diffs == 0] = 0.5  # i == j

    return F.binary_cross_entropy(pred_probs, true_probs)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

EPOCHS = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

RUN_NAME = "ranknet_v1_cpu_lite"
CHECKPOINT_PATH = f"best_lens_predictor_{RUN_NAME}.pth"

# =====================================================================
# 8. TRAINING & VALIDATION EXECUTION LOOPS
# =====================================================================
best_val_acc = 0.0

print(f"\nStarting 10-Epoch Training on {device.type.upper()}...")
for epoch in range(1, EPOCHS + 1):
    # --- Train Pass ---
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    for images, target_vectors in train_loader:
        images = images.to(device)
        target_vectors = target_vectors.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = ranknet_loss(outputs, target_vectors)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        _, true_best_param = torch.max(target_vectors, 1)

        total_train += target_vectors.size(0)
        correct_train += (predicted == true_best_param).sum().item()

    train_acc = 100 * correct_train / total_train
    train_loss = running_loss / total_train

    # --- Validation Pass ---
    model.eval()
    correct_val, total_val = 0, 0

    with torch.no_grad():
        for images, target_vectors in val_loader:
            images = images.to(device)
            target_vectors = target_vectors.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)
            _, true_best_param = torch.max(target_vectors, 1)

            total_val += target_vectors.size(0)
            correct_val += (predicted == true_best_param).sum().item()

    val_acc = 100 * correct_val / total_val
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT_PATH)

    print(
        f"Epoch [{epoch:02d}/{EPOCHS}] | "
        f"Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}%"
    )

print(f"\nTraining Complete.")
print(f"Best Validation Accuracy achieved: {best_val_acc:.2f}%")

Training executing on device: cuda

--- Initializing Training Dataset ---
Loaded 4,800 total images across 200 classes.
Pre-parsing target confidence vectors into memory...

--- Initializing Validation Dataset ---
Loaded 600 total images across 80 classes.
Pre-parsing target confidence vectors into memory...

Starting 10-Epoch Training on CUDA...


## CSA2

In [3]:
import os
import json
import urllib.request
from pathlib import Path
import random
import itertools

import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# CSA2 ALGORITHM FUNCTIONS
# ============================================================
def get_csa2_grid_assignments(K):
    param_indices = list(range(1, 28))
    coordinates = list(itertools.product([0, 1, 2], repeat=3))
    parameter_space = list(zip(param_indices, coordinates))
    
    grids = {}
    if 1 <= K <= 7:
        grids[0] = [idx for idx, _ in parameter_space]
    elif 8 <= K <= 26:
        for idx, (x, y, z) in parameter_space:
            grid_x = 0 if x < 1 else 1
            grid_y = 0 if y < 1 else 1
            grid_z = 0 if z < 1 else 1
            grid_id = (grid_x << 2) | (grid_y << 1) | grid_z
            if grid_id not in grids:
                grids[grid_id] = []
            grids[grid_id].append(idx)
    elif K == 27:
        for idx, _ in parameter_space:
            grids[idx] = [idx]
    else:
        raise ValueError("K must be between 1 and 27 inclusive.")
    return grids

def sample_csa2_parameters(K):
    grids = get_csa2_grid_assignments(K)
    grid_ids = list(grids.keys())
    print(f"grid_ids: {grid_ids}")
    
    random.shuffle(grid_ids)
    for gid in grid_ids:
        print(f"gids:{gid}")
        random.shuffle(grids[gid])
        
    selected_options = []
    grid_index = 0
    while len(selected_options) < K:
        current_grid_id = grid_ids[grid_index % len(grid_ids)]
        current_grid_pool = grids[current_grid_id]
        if len(current_grid_pool) > 0:
            chosen_param = current_grid_pool.pop(0)
            selected_options.append(chosen_param)
        grid_index += 1
    print(selected_options)
    return sorted(selected_options)


# ============================================================
# LOAD IMAGENET RESNET18 (final evaluation classifier)
# ============================================================
resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 200)

FINETUNED_MODEL_PATH = "resnet18_tinyimagenet_fc.pth" # <-- UPDATE THIS PATH
resnet.load_state_dict(torch.load(FINETUNED_MODEL_PATH, map_location=device))
resnet.to(device)
resnet.eval()
print("ImageNet ResNet loaded.")

# ============================================================
# IMAGE TRANSFORM
# ============================================================
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ============================================================
# LOAD TEST CSV - TAKE FIRST 600 IMAGES
# ============================================================
df = pd.read_csv("oracle_test_200 .csv")
df = df.iloc[:600].reset_index(drop=True)
print(f"Evaluating on {len(df)} test samples.")

# ============================================================
# CSA2 EVALUATION LOOP
# ============================================================
# Set your budget here! Change this to 1, 4, 8, 14, 27 to generate your graph data.
K_BUDGET = 6 

top1_correct = 0
top5_correct = 0
total_valid = 0

print(f"Evaluating CSA2 Algorithm with Budget K = {K_BUDGET}...")

for i, row in df.iterrows():
    csv_path = Path(row["image_path"])
    
    class_name = csv_path.parent.name
    filename = csv_path.name
    lighting = row["lighting"] 
    target = int(row["label"])
    base_dir = csv_path.parents[4]
    
    # 1. Ask CSA2 for K diverse parameters for this image
    selected_parameters = sample_csa2_parameters(K_BUDGET)
    
    # Flags to track if ANY of the K parameters get it right
    image_top1_success = False
    image_top5_success = False
    
    # 2. Loop through the K parameters
    for param_id in selected_parameters:
        param_path = Path(os.path.join('ImageNet-ES-Diverse/', base_dir, "param_control", lighting, f"param_{param_id}", class_name, filename))

        if not param_path.exists():
            continue 

        # Load chosen image and classify
        img = Image.open(param_path).convert("RGB")
        img_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = resnet(img_tensor)
            pred = logits.argmax(dim=1).item()
            
            # Check Top-1
            if pred == target:
                image_top1_success = True
                image_top5_success = True # If Top-1 is right, Top-5 is also right
                break # EARLY STOPPING: We found a winning image! No need to check the remaining K parameters.
                
            # Check Top-5 (if Top-1 failed)
            top5 = torch.topk(logits, k=5, dim=1).indices.squeeze(0)
            if target in top5:
                image_top5_success = True
                
    # Update global counters if the image was successfully classified under this budget
    if image_top1_success:
        top1_correct += 1
    if image_top5_success:
        top5_correct += 1

    total_valid += 1
    if total_valid % 50 == 0:
        print(f"Processed {total_valid} valid images...")

# ============================================================
# RESULTS
# ============================================================
print("\n===============================")
print(f"CSA2 Pipeline Results (Budget K={K_BUDGET})")
print("===============================")
if total_valid == 0:
    print("ERROR: 0 images evaluated. Check if your param_path exists on your hard drive!")
else:
    print(f"Images Evaluated : {total_valid}")
    print(f"Top-1 Accuracy   : {100 * top1_correct / total_valid:.2f}%")
    print(f"Top-5 Accuracy   : {100 * top5_correct / total_valid:.2f}%")

Using device: cuda
ImageNet ResNet loaded.
Evaluating on 600 test samples.
Evaluating CSA2 Algorithm with Budget K = 6...
grid_ids: [0]
gids:0
[5, 27, 24, 22, 15, 11]
grid_ids: [0]
gids:0
[18, 16, 27, 6, 15, 22]
grid_ids: [0]
gids:0
[3, 8, 14, 16, 13, 24]
grid_ids: [0]
gids:0
[4, 11, 7, 3, 22, 5]
grid_ids: [0]
gids:0
[14, 6, 26, 11, 17, 23]
grid_ids: [0]
gids:0
[27, 25, 9, 15, 8, 17]
grid_ids: [0]
gids:0
[14, 19, 15, 27, 9, 17]
grid_ids: [0]
gids:0
[17, 14, 23, 9, 10, 21]
grid_ids: [0]
gids:0
[12, 1, 7, 25, 17, 10]
grid_ids: [0]
gids:0
[3, 18, 11, 16, 21, 8]
grid_ids: [0]
gids:0
[23, 10, 12, 16, 19, 24]
grid_ids: [0]
gids:0
[21, 11, 27, 16, 2, 23]
grid_ids: [0]
gids:0
[27, 23, 22, 26, 15, 4]
grid_ids: [0]
gids:0
[4, 18, 11, 17, 15, 1]
grid_ids: [0]
gids:0
[19, 15, 14, 21, 9, 6]
grid_ids: [0]
gids:0
[17, 25, 9, 13, 15, 26]
grid_ids: [0]
gids:0
[22, 13, 5, 11, 24, 9]
grid_ids: [0]
gids:0
[2, 16, 8, 12, 20, 24]
grid_ids: [0]
gids:0
[5, 8, 25, 14, 22, 11]
grid_ids: [0]
gids:0
[4, 6, 1, 7, 